# 02 — Feature Engineering

Constructs the feature set for both train and val.  
**Never touches test_locked.parquet.**

Key features engineered:
- Velocity features (transaction count over time windows)
- Behavioural deviation (amount vs user's historical mean)
- Temporal features (hour, day, is_night, is_weekend)
- Email domain risk flags
- Rapid repeat transaction detection

In [ ]:
!pip install -q lightgbm shap category_encoders

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings('ignore')

train = pd.read_parquet('/kaggle/working/train.parquet')
val   = pd.read_parquet('/kaggle/working/val.parquet')

with open('/kaggle/working/cols_to_drop.json') as f:
    cols_to_drop = json.load(f)

print(f'Train: {train.shape}, Val: {val.shape}')

In [ ]:
START_DATE = pd.Timestamp('2017-12-01')

def add_temporal_features(df):
    dt = START_DATE + pd.to_timedelta(df['TransactionDT'], unit='s')
    df['hour_of_day'] = dt.dt.hour
    df['day_of_week'] = dt.dt.dayofweek
    df['is_night'] = ((df['hour_of_day'] < 6) | (df['hour_of_day'] >= 23)).astype(int)
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    return df

train = add_temporal_features(train)
val   = add_temporal_features(val)
print('Temporal features added')

In [ ]:
def add_amount_features(df):
    df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])

    # Amount vs card's historical mean (computed on train, applied to val)
    card_mean = train.groupby('card1')['TransactionAmt'].mean().rename('card1_amt_mean')
    df = df.merge(card_mean, on='card1', how='left')
    df['amt_vs_card_mean'] = df['TransactionAmt'] / df['card1_amt_mean'].fillna(df['TransactionAmt'])

    return df

train = add_amount_features(train)
val   = add_amount_features(val)
print('Amount features added')

In [ ]:
# Velocity features: how many transactions per card in last N hours
# Approximated using transaction count within sorted windows (no exact timestamps)
# In production this would use a feature store / Redis

def add_velocity_features(df):
    # Count of transactions per card — rolling approximation
    df_sorted = df.sort_values('TransactionDT')

    # Transactions per card total (proxy for velocity)
    card_txn_count = df_sorted.groupby('card1').cumcount()
    df['card1_txn_count'] = card_txn_count.values

    # Time since last transaction for same card
    df_sorted['prev_dt'] = df_sorted.groupby('card1')['TransactionDT'].shift(1)
    df_sorted['time_since_last_txn'] = (df_sorted['TransactionDT'] - df_sorted['prev_dt']) / 3600
    df['time_since_last_txn'] = df_sorted['time_since_last_txn'].values
    df['rapid_repeat'] = (df['time_since_last_txn'] < 0.1).astype(int)

    return df

train = add_velocity_features(train)
val   = add_velocity_features(val)
print('Velocity features added')

In [ ]:
HIGH_RISK_DOMAINS = {'anonymous.com', 'aim.com', 'protonmail.com'}

def add_email_features(df):
    df['P_emaildomain'] = df['P_emaildomain'].fillna('unknown')
    df['R_emaildomain'] = df['R_emaildomain'].fillna('unknown')
    df['p_email_is_high_risk'] = df['P_emaildomain'].isin(HIGH_RISK_DOMAINS).astype(int)
    df['r_email_is_high_risk'] = df['R_emaildomain'].isin(HIGH_RISK_DOMAINS).astype(int)
    df['email_domain_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
    return df

train = add_email_features(train)
val   = add_email_features(val)
print('Email features added')

In [ ]:
# Drop high-missing columns identified in EDA
cols_to_drop_present = [c for c in cols_to_drop if c in train.columns]
train = train.drop(columns=cols_to_drop_present)
val   = val.drop(columns=cols_to_drop_present)

# Drop non-numeric identifiers
id_cols = ['TransactionID', 'TransactionDT']
train = train.drop(columns=[c for c in id_cols if c in train.columns])
val   = val.drop(columns=[c for c in id_cols if c in val.columns])

print(f'After dropping: train {train.shape}, val {val.shape}')

In [ ]:
# Label encode remaining categoricals
from sklearn.preprocessing import LabelEncoder
import pickle

cat_cols = train.select_dtypes(include='object').columns.tolist()
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    val[col]   = le.transform(val[col].astype(str).map(
        lambda x: x if x in le.classes_ else le.classes_[0]
    ))
    encoders[col] = le

with open('/kaggle/working/label_encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

print(f'Encoded {len(cat_cols)} categorical columns')

In [ ]:
# Fill remaining NaNs with -999 (LightGBM handles this natively)
train = train.fillna(-999)
val   = val.fillna(-999)

# Save feature names
feature_cols = [c for c in train.columns if c != 'isFraud']

train.to_parquet('/kaggle/working/train_features.parquet', index=False)
val.to_parquet('/kaggle/working/val_features.parquet', index=False)

with open('/kaggle/working/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

print(f'Feature engineering complete. {len(feature_cols)} features saved.')
print('Saved: train_features.parquet, val_features.parquet, feature_cols.json')